# Preparacion del Dataset Logistico

Notebook dedicado a transformar la muestra base de FreshRetailNet en los archivos `dataset_logistico.csv` y `dataset_logistico_app.csv`. No incluye analisis EDA; su funcion es dejar los datos listos para limpieza, modelado y uso en la app.

## 1. Librerias

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## 2. Carga de la muestra base

In [ ]:
RUTA_BASE = "../data/freshretailnet_muestra_100k_secuencial.csv"

df_raw = pd.read_csv(RUTA_BASE)

print(f"Filas: {df_raw.shape[0]:,}")
print(f"Columnas: {df_raw.shape[1]:,}")
display(df_raw.head())

## 3. Seleccion de columnas base

In [ ]:
COLUMNAS_BASE = [
    "dt",
    "product_id",
    "store_id",
    "first_category_id",
    "second_category_id",
    "third_category_id",
    "sale_amount",
    "stock_hour6_22_cnt",
    "hours_stock_status",
    "discount",
    "holiday_flag",
    "activity_flag",
    "nombre_producto",
    "nombre_tienda",
    "nombre_categoria_n1",
    "nombre_categoria_n2",
    "nombre_categoria_n3",
]

columnas_existentes = [col for col in COLUMNAS_BASE if col in df_raw.columns]
df = df_raw[columnas_existentes].copy()

df["dt"] = pd.to_datetime(df["dt"], errors="coerce")
display(df.head())

## 4. Catalogo demostrativo para la app

In [ ]:
catalogo_productos_app = [
    ("Metotrexato 500mg", "Oncologicos"),
    ("Ciclofosfamida 1g", "Oncologicos"),
    ("Vincristina 1mg", "Oncologicos"),
    ("Doxorrubicina 50mg", "Oncologicos"),
    ("Ondansetron 8mg", "Antiemeticos"),
    ("Dexametasona 4mg", "Antiemeticos"),
    ("Suero Fisiologico 500ml", "Soporte"),
    ("Dextrosa al 5%", "Soporte"),
    ("Cloruro de Potasio 20%", "Soporte"),
    ("Cateter Endovenoso 24G", "Insumos"),
    ("Cateter Port-a-Cath", "Insumos"),
    ("Jeringa 5ml", "Insumos"),
    ("Equipo de Venoclisis", "Insumos"),
    ("Llave de 3 vias", "Insumos"),
    ("Pediasure Plus 200ml", "Nutricion"),
    ("Formula Polimerica", "Nutricion"),
    ("Espesante de Alimentos", "Nutricion"),
    ("Mascarilla N95 Pediatrica", "Higiene"),
    ("Alcohol en Gel 70%", "Higiene"),
    ("Guantes Esteriles", "Higiene"),
    ("Jabon Clorhexidina", "Higiene"),
    ("Tubo Vacutainer (Rojo)", "Laboratorio"),
    ("Tubo Vacutainer (Lila)", "Laboratorio"),
    ("Placas de Rayos X", "Imagenes"),
    ("Crema de Hidrocortisona", "Topicos"),
    ("Gasas Esteriles", "Curacion"),
    ("Esparadrapo Micropore", "Curacion"),
    ("Paracetamol Jarabe", "Analgesicos"),
    ("Ibuprofeno Pediatrico", "Analgesicos"),
    ("Ceftriaxona 1g", "Antibioticos"),
    ("Arroz extra", "Canasta basica - Cereales"),
    ("Fideos", "Canasta basica - Cereales"),
    ("Avena", "Canasta basica - Cereales"),
    ("Pan frances", "Canasta basica - Panaderia"),
    ("Harina de trigo", "Canasta basica - Cereales"),
    ("Papa blanca", "Canasta basica - Tuberculos"),
    ("Camote", "Canasta basica - Tuberculos"),
    ("Yuca", "Canasta basica - Tuberculos"),
    ("Lenteja", "Canasta basica - Menestras"),
    ("Frejol canario", "Canasta basica - Menestras"),
    ("Arveja partida", "Canasta basica - Menestras"),
    ("Aceite vegetal", "Canasta basica - Aceites"),
    ("Azucar rubia", "Canasta basica - Abarrotes"),
    ("Sal yodada", "Canasta basica - Abarrotes"),
    ("Leche evaporada", "Canasta basica - Lacteos"),
    ("Queso fresco", "Canasta basica - Lacteos"),
    ("Huevo de gallina", "Canasta basica - Proteinas"),
    ("Pollo entero", "Canasta basica - Carnes"),
    ("Pechuga de pollo", "Canasta basica - Carnes"),
    ("Carne de res", "Canasta basica - Carnes"),
    ("Carne de cerdo", "Canasta basica - Carnes"),
    ("Pescado bonito", "Canasta basica - Pescados"),
    ("Jurel", "Canasta basica - Pescados"),
    ("Atun en conserva", "Canasta basica - Conservas"),
    ("Sardina en conserva", "Canasta basica - Conservas"),
    ("Cebolla roja", "Canasta basica - Verduras"),
    ("Tomate", "Canasta basica - Verduras"),
    ("Zanahoria", "Canasta basica - Verduras"),
    ("Zapallo", "Canasta basica - Verduras"),
    ("Lechuga", "Canasta basica - Verduras"),
    ("Platano de seda", "Canasta basica - Frutas"),
    ("Manzana", "Canasta basica - Frutas"),
    ("Naranja", "Canasta basica - Frutas"),
    ("Limon", "Canasta basica - Frutas"),
    ("Agua embotellada", "Canasta basica - Bebidas"),
]

productos_para_catalogo = (
    df.groupby("product_id", observed=True)["sale_amount"]
    .sum()
    .sort_values(ascending=False)
    .head(len(catalogo_productos_app))
    .index
    .tolist()
)

catalogo_app = pd.DataFrame(catalogo_productos_app, columns=["nombre_producto_app", "categoria_app"])
catalogo_app.insert(0, "product_id_original", productos_para_catalogo)
catalogo_app["product_id_original"] = catalogo_app["product_id_original"].astype(str)
catalogo_app.insert(1, "product_id_app", [f"{i:03d}" for i in range(1, len(catalogo_app) + 1)])

df["product_id_original"] = df["product_id"].astype(str)
df = df.merge(catalogo_app, on="product_id_original", how="left")
df["tipo_producto_app"] = np.select(
    [
        df["categoria_app"].fillna("").str.startswith("Canasta basica"),
        df["categoria_app"].notna(),
    ],
    ["Alimento", "Clinico"],
    default="Producto general",
)

def crear_nombre_presentable(codigo, tipo):
    if tipo == "Alimento":
        return f"Alimento {codigo}"
    if tipo == "Clinico":
        return f"Insumo {codigo}"
    return f"Producto general {codigo}"

df["name_products"] = df["nombre_producto_app"].fillna(df["nombre_producto"])
df["nombre_categoria_n1"] = df["categoria_app"].fillna(df["nombre_categoria_n1"])

mapa_ids_genericos = {
    valor: f"R{idx:03d}"
    for idx, valor in enumerate(
        df.loc[df["product_id_app"].isna(), "product_id_original"].drop_duplicates(),
        start=len(catalogo_app) + 1,
    )
}

df["product_id"] = df["product_id_app"].fillna(df["product_id_original"].map(mapa_ids_genericos))
df["name_products"] = df["name_products"].fillna(
    df.apply(lambda fila: crear_nombre_presentable(fila["product_id"], fila["tipo_producto_app"]), axis=1)
)
df["product_id"] = df["product_id"].astype("category")

df = df.drop(columns=["nombre_producto_app", "categoria_app", "product_id_app"])
display(df.head())

## 5. Creacion de variables temporales y dataset final

In [ ]:
df = df.sort_values(["store_id", "product_id", "dt"]).reset_index(drop=True)

df["anio"] = df["dt"].dt.year
df["mes"] = df["dt"].dt.month
df["dia_mes"] = df["dt"].dt.day
df["dia_semana"] = df["dt"].dt.dayofweek
df["fin_semana"] = (df["dia_semana"] >= 5).astype(int)

grupo = df.groupby(["store_id", "product_id"], observed=True)
df["venta_lag_1"] = grupo["sale_amount"].shift(1)
df["venta_lag_7"] = grupo["sale_amount"].shift(7)
df["venta_promedio_7d"] = grupo["sale_amount"].transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
df["venta_promedio_14d"] = grupo["sale_amount"].transform(lambda s: s.shift(1).rolling(14, min_periods=1).mean())
df["stock_lag_1"] = grupo["stock_hour6_22_cnt"].shift(1)

import re
def contar_horas_con_stock(valor):
    if pd.isna(valor):
        return np.nan
    numeros = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", str(valor))
    if not numeros:
        return np.nan
    valores = np.array([float(numero) for numero in numeros])
    return np.sum(valores > 0)

df["horas_con_stock"] = df["hours_stock_status"].apply(contar_horas_con_stock)

columnas_dataset_logistico = [
    "dt", "sale_amount", "product_id", "store_id",
    "first_category_id", "second_category_id", "third_category_id",
    "discount", "holiday_flag", "activity_flag",
    "mes", "dia_mes", "dia_semana", "fin_semana",
    "horas_con_stock", "venta_lag_1", "venta_lag_7",
    "venta_promedio_7d", "venta_promedio_14d", "stock_lag_1"
]

columnas_app = [
    "dt", "sale_amount", "product_id", "product_id_original", "name_products",
    "tipo_producto_app", "store_id", "nombre_tienda", "nombre_categoria_n1",
    "discount", "holiday_flag", "activity_flag", "horas_con_stock",
    "venta_lag_1", "venta_lag_7", "venta_promedio_7d", "venta_promedio_14d", "stock_lag_1"
]

df_logistico = df[[col for col in columnas_dataset_logistico if col in df.columns]].copy()
df_logistico_app = df[[col for col in columnas_app if col in df.columns]].copy()

print(f"Filas dataset logistico: {len(df_logistico):,}")
print(f"Filas dataset app: {len(df_logistico_app):,}")
display(df_logistico.head())
display(df_logistico_app.head())

## 6. Exportacion de CSV

In [ ]:
ruta_modelado = "../data/dataset_logistico.csv"
ruta_app = "../data/dataset_logistico_app.csv"

df_logistico.to_csv(ruta_modelado, index=False, encoding="utf-8")
df_logistico_app.to_csv(ruta_app, index=False, encoding="utf-8")

print(f"CSV de modelado guardado en: {ruta_modelado}")
print(f"Filas modelado: {len(df_logistico):,}")
print(f"CSV para app guardado en: {ruta_app}")
print(f"Filas app: {len(df_logistico_app):,}")